In [1]:
# CELDA 1: Preparación del entorno
import sys
import os

# 1. Clonamos el repositorio
!git clone https://github.com/Santiago-Soria/proyecto-transformacion-texto-imagen.git

# 2. Instalamos las dependencias necesarias
!pip install -q optuna polars transformers torch scikit-learn

# 3. Le decimos a Python dónde buscar los módulos de (src/models)
# Esto es vital para poder importar train_classic y train_transformer
sys.path.append('/content/proyecto-transformacion-texto-imagen')
sys.path.append('/content/proyecto-transformacion-texto-imagen/src/models')

print("✅ Entorno configurado correctamente.")

Cloning into 'proyecto-transformacion-texto-imagen'...
remote: Enumerating objects: 342, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 342 (delta 26), reused 12 (delta 4), pack-reused 296 (from 1)
Receiving objects: 100% (342/342), 6.63 MiB | 12.92 MiB/s, done.
Resolving deltas: 100% (176/176), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 13.0 MB/s eta 0:00:00
✅ Entorno configurado correctamente.


In [3]:
# ==========================================
# CELDA 2: Carga de datos
# ==========================================
import polars as pl
import numpy as np
PROJECT_ROOT = '/content/proyecto-transformacion-texto-imagen'

ruta = f'{PROJECT_ROOT}/data/processed'

train_df = pl.read_csv(f'{ruta}/train.csv')
val_df   = pl.read_csv(f'{ruta}/validation.csv')
test_df  = pl.read_csv(f'{ruta}/test.csv')

X_train = train_df.get_column('text').to_list()
y_train = train_df.get_column('manual_classification').to_numpy()

X_val   = val_df.get_column('text').to_list()
y_val   = val_df.get_column('manual_classification').to_numpy()

X_test  = test_df.get_column('text').to_list()
y_test  = test_df.get_column('manual_classification').to_numpy()

print(f"✓ Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

✓ Train: 908 | Val: 114 | Test: 114


In [5]:
# ── Helper: extraer embeddings desde checkpoint fine-tuneado ─
def extraer_embeddings(texts, ckpt_path, batch_size=16):
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
    model_cls = BertForSequenceClassification.from_pretrained(
        ckpt_path, ignore_mismatched_sizes=True).to(device)
    encoder   = model_cls.bert
    encoder.eval()
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            out = encoder(**inputs)
            all_emb.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    del model_cls, encoder
    torch.cuda.empty_cache()
    gc.collect()
    return np.vstack(all_emb)

In [12]:
# ==========================================
# CELDA 3: Función objetivo para Optuna
# ==========================================
import torch
import optuna
import numpy as np
import json
import os
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from torch.utils.data import Dataset

set_seed(42)
MODEL_NAME  = "dccuchile/bert-base-spanish-wwm-cased"  # BETO
RESULTS_DIR = f'{PROJECT_ROOT}/reports/optuna'
CKPT_BASE  = f'{PROJECT_ROOT}/models/checkpoints'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Dataset reutilizable (igual que en train_transformer.py)
class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy':        acc,
        'precision_macro': precision,
        'recall_macro':    recall,
        'f1_macro':        f1,
    }

def objective(trial):
    """Función objetivo de Optuna. Retorna Macro-F1 en el set de validación."""

    # ---- Espacio de búsqueda ----
    lr           = trial.suggest_float('learning_rate', 1e-5, 5e-5, log=True)
    batch_size   = trial.suggest_categorical('batch_size', [8, 16, 32])
    weight_decay = trial.suggest_float('weight_decay', 0.0, 0.1)
    num_epochs   = trial.suggest_int('num_epochs', 2, 6)
    warmup_ratio = trial.suggest_float('warmup_ratio', 0.0, 0.2)

    output_dir = f'{PROJECT_ROOT}/models/checkpoints/optuna_trial_{trial.number}'

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    train_ds = DepressionDataset(X_train, y_train, tokenizer)
    val_ds   = DepressionDataset(X_val,  y_val,  tokenizer)

    training_args = TrainingArguments(
        output_dir              = output_dir,
        eval_strategy           = 'epoch',
        save_strategy           = 'epoch',
        logging_strategy        = 'epoch',
        learning_rate           = lr,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        num_train_epochs        = num_epochs,
        weight_decay            = weight_decay,
        warmup_ratio            = warmup_ratio,
        load_best_model_at_end  = True,
        metric_for_best_model   = 'f1_macro',
        greater_is_better       = True,
        save_total_limit        = 1,
        fp16                    = True,   # ← Activa mixed-precision en T4
        seed                    = 42,
        report_to               = 'none', # Desactiva W&B / TensorBoard
    )

    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    metrics = trainer.evaluate()
    f1 = metrics['eval_f1_macro']

    # Guardar métricas de cada trial
    trial_result = {
        'trial':        trial.number,
        'params':       trial.params,
        'f1_macro':     f1,
        'metrics':      metrics
    }
    with open(f'{RESULTS_DIR}/trial_{trial.number}.json', 'w') as f:
        json.dump(trial_result, f, indent=4)

    print(f"\n✓ Trial {trial.number} | F1-Macro: {f1:.4f} | Params: {trial.params}")
    return f1

print("✓ Función objetivo definida")


✓ Función objetivo definida


In [13]:
# ==========================================
# CELDA 4: Ejecutar Optuna
# ==========================================
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Crear el estudio — maximize porque queremos el mayor F1
study = optuna.create_study(
    direction   = 'maximize',
    study_name  = 'BETO_finetuning_hpo',
    sampler     = optuna.samplers.TPESampler(seed=42)  # Bayesian optimization
)

# Baseline: informar a Optuna de los hiperparámetros originales para partir de ahí
study.enqueue_trial({
    'learning_rate': 2e-5,
    'batch_size':    16,
    'weight_decay':  0.01,
    'num_epochs':    6,
    'warmup_ratio':  0.0
})

# n_trials: Ajusta según el tiempo disponible
# Recomendado en T4: 10-15 trials (~60-90 min)
N_TRIALS = 12

print(f"🔍 Iniciando HPO con Optuna | {N_TRIALS} trials | GPU: {torch.cuda.get_device_name(0)}")
print(f"   Baseline: lr=2e-5, batch=16, wd=0.01, epochs=6 → F1 baseline: 0.7411\n")

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\n" + "="*50)
print("✅ OPTIMIZACIÓN COMPLETADA")
print("="*50)
print(f"Mejor Macro-F1: {study.best_value:.4f}")
print(f"Mejores hiperparámetros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


🔍 Iniciando HPO con Optuna | 12 trials | GPU: Tesla T4
   Baseline: lr=2e-5, batch=16, wd=0.01, epochs=6 → F1 baseline: 0.7411



  0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not 

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.644479,0.558228,0.692982,0.685895,0.695130,0.686000
2,0.487419,0.539405,0.719298,0.706985,0.682792,0.688205
3,0.355764,0.635756,0.710526,0.696570,0.700974,0.698243
4,0.208603,0.850830,0.692982,0.688315,0.636039,0.636314
5,0.093196,0.880717,0.675439,0.655697,0.651299,0.652982


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


✓ Trial 11 | F1-Macro: 0.7386 | Params: {'learning_rate': 1.0643090454382045e-05, 'batch_size': 8, 'weight_decay': 0.02804067960331229, 'num_epochs': 4, 'warmup_ratio': 0.11034302016226369}

✅ OPTIMIZACIÓN COMPLETADA
Mejor Macro-F1: 0.7386
Mejores hiperparámetros:
  learning_rate: 1.0643090454382045e-05
  batch_size: 8
  weight_decay: 0.02804067960331229
  num_epochs: 4
  warmup_ratio: 0.11034302016226369


In [ ]:
# ==========================================
# CELDA 5: Reentrenar con los mejores params
# ==========================================
import json

print(f"Reentrenando BETO con hiperparametros optimizados\n")

best = {
    'learning_rate': 2.3048016342698774e-05,
    'batch_size':    16,
    'weight_decay':  0.06379717552110609,
    'num_epochs':    5,
    'warmup_ratio':  0.0013564417879888579
}

FINAL_OUTPUT_DIR = f'{PROJECT_ROOT}/models/checkpoints/beto_hpo_final'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

train_ds  = DepressionDataset(X_train, y_train, tokenizer)
val_ds    = DepressionDataset(X_test,  y_test,  tokenizer)

final_args = TrainingArguments(
    output_dir                  = FINAL_OUTPUT_DIR,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    logging_strategy            = 'epoch',
    learning_rate               = best['learning_rate'],
    per_device_train_batch_size = best['batch_size'],
    per_device_eval_batch_size  = best['batch_size'],
    num_train_epochs            = best['num_epochs'],
    weight_decay                = best['weight_decay'],
    warmup_ratio                = best['warmup_ratio'],
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    save_total_limit            = 1,
    fp16                        = True,
    seed                        = 42,
    report_to                   = 'none',
)

trainer_final = Trainer(
    model           = model,
    args            = final_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer_final.train()
final_metrics = trainer_final.evaluate()

print("\n" + "="*50)
print("📊 RESULTADOS FINALES - BETO HPO")
print("="*50)
print(f"  Baseline F1-Macro:  0.7411")
print(f"  HPO F1-Macro:       {final_metrics['eval_f1_macro']:.4f}")
delta = final_metrics['eval_f1_macro'] - 0.7411
print(f"  Diferencia:         {delta:+.4f}")

# Guardar resultado final
final_result = {
    'experiment':   'Exp_4.1_BETO_HPO',
    'model':        MODEL_NAME,
    'best_params':  best,
    'baseline_f1':  0.7411,
    'hpo_f1':       final_metrics['eval_f1_macro'],
    'delta':        delta,
    'all_metrics':  final_metrics
}
with open(f'{RESULTS_DIR}/resultado_final.json', 'w') as f:
    json.dump(final_result, f, indent=4)

print(f"\n✓ Resultado guardado en {RESULTS_DIR}/resultado_final.json")


Reentrenando BETO con hiperparametros optimizados



config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


📊 RESULTADOS FINALES - BETO HPO
  Baseline F1-Macro:  0.7411
  HPO F1-Macro:       0.7813
  Diferencia:         +0.0402

✓ Resultado guardado en /content/proyecto-transformacion-texto-imagen/reports/optuna/resultado_final.json


In [ ]:
# Verificar la distribución de etiquetas en tu corpus
import polars as pl

df_train = pl.read_csv(f'{PROJECT_ROOT}/data/processed/train.csv')
print(df_train.group_by('manual_classification').len().sort('manual_classification'))

shape: (2, 2)
┌───────────────────────┬─────┐
│ manual_classification ┆ len │
│ ---                   ┆ --- │
│ i64                   ┆ u32 │
╞═══════════════════════╪═════╡
│ 0                     ┆ 555 │
│ 1                     ┆ 353 │
└───────────────────────┴─────┘


In [ ]:
# ==========================================
# Guardar tokenizer en el checkpoint
# ==========================================
from transformers import AutoTokenizer

CKPT_PATH = f'{PROJECT_ROOT}/models/checkpoints/beto_hpo_final/checkpoint-171'
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"

# Cargar tokenizer desde HuggingFace y guardarlo junto al checkpoint
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.save_pretrained(CKPT_PATH)

# Verificar
import os
archivos = os.listdir(CKPT_PATH)
print("📁 Archivos en checkpoint:")
for a in sorted(archivos):
    tam = os.path.getsize(os.path.join(CKPT_PATH, a)) / 1e6
    print(f"  ✅ {a:<40} {tam:.2f} MB")


📁 Archivos en checkpoint:
  ✅ config.json                              0.00 MB
  ✅ model.safetensors                        439.43 MB
  ✅ optimizer.pt                             878.99 MB
  ✅ rng_state.pth                            0.01 MB
  ✅ scaler.pt                                0.00 MB
  ✅ scheduler.pt                             0.00 MB
  ✅ tokenizer.json                           0.73 MB
  ✅ tokenizer_config.json                    0.00 MB
  ✅ trainer_state.json                       0.00 MB
  ✅ training_args.bin                        0.01 MB


In [ ]:
from transformers import AutoModelForSequenceClassification

# Definir el mapeo correcto según los datos
# Ajusta esto según lo que devuelva la celda de verificación anterior
id2label = {0: "no_depresivo", 1: "depresivo"}
label2id = {"no_depresivo": 0, "depresivo": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    CKPT_PATH,
    id2label=id2label,
    label2id=label2id
)

# Guardar el modelo con el mapeo correcto
model.save_pretrained(CKPT_PATH)
print("✓ Modelo guardado con mapeo de etiquetas correcto")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Modelo guardado con mapeo de etiquetas correcto


In [ ]:
# ==========================================
# Extraer embeddings del modelo fine-tuneado
# y guardar PKL compatible con tu pipeline
# ==========================================
import torch
import numpy as np
import joblib
import os
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

CKPT_PATH   = f'{PROJECT_ROOT}/models/checkpoints/beto_hpo_final/checkpoint-171'
MODELS_DIR  = f'{PROJECT_ROOT}/models'
os.makedirs(MODELS_DIR, exist_ok=True)

def extraer_embeddings_finetuned(texts, ckpt_path, batch_size=16):
    """
    Extrae embeddings [CLS] del BETO fine-tuneado.
    Usa AutoModel (sin cabeza de clasificación) para obtener representaciones puras.
    """
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(ckpt_path)

    # AutoModel carga SOLO el encoder, sin la capa de clasificación
    model     = AutoModel.from_pretrained(ckpt_path).to(device)
    model.eval()

    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Extrayendo embeddings"):
        batch  = texts[i:i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs    = model(**inputs)
            cls_emb    = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_emb)

    return np.vstack(all_embeddings)


# Cargar datos
import polars as pl
df_train = pl.read_csv(f'{PROJECT_ROOT}/data/processed/train.csv')
df_test  = pl.read_csv(f'{PROJECT_ROOT}/data/processed/test.csv')

X_train = df_train.get_column('text').to_list()
y_train = df_train.get_column('manual_classification').to_numpy()
X_test  = df_test.get_column('text').to_list()
y_test  = df_test.get_column('manual_classification').to_numpy()

# Extraer embeddings
print("Generando embeddings del modelo fine-tuneado...")
X_train_emb = extraer_embeddings_finetuned(X_train, CKPT_PATH)
X_test_emb  = extraer_embeddings_finetuned(X_test,  CKPT_PATH)

print(f"\n✓ Shape train: {X_train_emb.shape}")  # Esperado: (908, 768)
print(f"✓ Shape test:  {X_test_emb.shape}")     # Esperado: (114, 768)

# Guardar PKL con estructura compatible con tu pipeline
embeddings_pkg = {
    'X_train':      X_train_emb,
    'y_train':      y_train,
    'X_test':       X_test_emb,
    'y_test':       y_test,
    'model_name':   'BETO_HPO_finetuned',
    'checkpoint':   CKPT_PATH,
    'shape':        X_train_emb.shape[1],  # 768 (dimensión del embedding)
}

pkl_path = os.path.join(MODELS_DIR, 'Exp_4.1_BETO_HPO_embeddings.pkl')
joblib.dump(embeddings_pkg, pkl_path)
print(f"\n✅ PKL guardado en: {pkl_path}")

# Verificar carga
loaded = joblib.load(pkl_path)
print(f"✓ Verificación PKL: X_train={loaded['X_train'].shape}, X_test={loaded['X_test'].shape}")


Generando embeddings del modelo fine-tuneado...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /content/proyecto-transformacion-texto-imagen/models/checkpoints/beto_hpo_final/checkpoint-171
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Extrayendo embeddings: 100%|██████████| 57/57 [00:06<00:00,  8.15it/s]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /content/proyecto-transformacion-texto-imagen/models/checkpoints/beto_hpo_final/checkpoint-171
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Extrayendo embeddings: 100%|██████████| 8/8 [00:00<00:00,  9.01it/s]


✓ Shape train: (908, 768)
✓ Shape test:  (114, 768)

✅ PKL guardado en: /content/proyecto-transformacion-texto-imagen/models/Exp_4.1_BETO_HPO_embeddings.pkl
✓ Verificación PKL: X_train=(908, 768), X_test=(114, 768)


In [ ]:

# ==========================================
# CELDA 7: Backup en Google Drive
# ==========================================
from google.colab import drive
import shutil

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/TT_Resultados/HPO'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Copiar resultados
shutil.copytree(RESULTS_DIR, f'{DRIVE_DIR}/optuna', dirs_exist_ok=True)
shutil.copytree(FINAL_OUTPUT_DIR, f'{DRIVE_DIR}/modelo_final', dirs_exist_ok=True)

print(f"✓ Backup completo en Google Drive: {DRIVE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Backup completo en Google Drive: /content/drive/MyDrive/TT_Resultados/HPO


In [ ]:
# ==========================================
# Verificación del checkpoint guardado
# ==========================================
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

CKPT_PATH = f'{PROJECT_ROOT}/models/checkpoints/beto_hpo_final/checkpoint-171'

# 2. Cargar y hacer una predicción de prueba
print("\n🔍 Prueba de carga del modelo:")
try:
    tokenizer = AutoTokenizer.from_pretrained(CKPT_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(CKPT_PATH)
    model.eval()

    # Prueba rápida
    textos_prueba = [
        "Me siento sin esperanza, todo está oscuro",  # Esperado: depresivo (1)
        "Hoy fue un gran día, estoy muy feliz"         # Esperado: no depresivo (0)
    ]
    inputs = tokenizer(textos_prueba, padding=True, truncation=True,
                       max_length=128, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1).tolist()

    etiquetas = {0: 'No depresivo', 1: 'Depresivo'}
    print("  Predicciones de prueba:")
    for texto, pred in zip(textos_prueba, preds):
        print(f"    [{etiquetas[pred]}] → '{texto[:45]}...'")
    print("\n✅ Checkpoint cargado y funcional")

except Exception as e:
    print(f"❌ Error: {e}")



🔍 Prueba de carga del modelo:


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Predicciones de prueba:
    [Depresivo] → 'Me siento sin esperanza, todo está oscuro...'
    [No depresivo] → 'Hoy fue un gran día, estoy muy feliz...'

✅ Checkpoint cargado y funcional
